In [ ]:
# --- ANÁLISE ESTATÍSTICA: IMPORTS ---
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.model_utils import (
    teste_shapiro,
    correlacao_spearman,
    plotar_heatmap_spearman,
    teste_kruskal_wallis_por_tercis,
    plotar_boxplot_kruskal,
 )

PROCESSED = PROJECT_ROOT / 'data' / 'processed'
df_final = pd.read_parquet(PROCESSED / 'base_diamante_es_vfinal.parquet')
print(f"✅ Dados carregados: {df_final.shape}")

# Estilo dos gráficos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print("✅ Bibliotecas de análise carregadas.")

In [ ]:
# --- TESTE DE NORMALIDADE (Shapiro-Wilk) ---
# Usamos uma amostra de até 5000 linhas (o teste tem limite)

colunas_teste = [
    'indice_atendimento_total_agua',
    'indice_tratamento_esgoto',
    'Taxa_Morbidade_100k_Hab',
    'RISCO_SOCIAL_FINAL'
 ]

resultado_shapiro = teste_shapiro(df_final, colunas_teste)

print("📊 TESTE DE NORMALIDADE (Shapiro-Wilk)\n")
print(f"{'Coluna':<45} {'p-valor':>10}  {'Distribuição'}")
print("-" * 70)

for row in resultado_shapiro.itertuples(index=False):
    if pd.isna(row.p_valor):
        p_txt = "      nan"
        resultado = "⚠️ Sem dados"
    else:
        p_txt = f"{row.p_valor:>10.4f}"
        resultado = "✅ Normal" if row.distribuicao == 'normal' else "❌ Não-normal → usar Spearman"
    print(f"{row.coluna:<45} {p_txt}  {resultado}")

In [ ]:
# --- CORRELAÇÃO DE SPEARMAN: SANEAMENTO vs SAÚDE ---

colunas_corr = [
    'indice_atendimento_total_agua',
    'indice_atendimento_esgoto_agua',
    'indice_tratamento_esgoto',
    'indice_perda_distribuicao_agua',
    'Taxa_Morbidade_100k_Hab',
    'RISCO_SOCIAL_FINAL',
    'vazio_sanitario',
 ]

corr_df = correlacao_spearman(df_final, colunas_corr)
plotar_heatmap_spearman(
    corr_df,
    titulo='Correlação de Spearman — Saneamento vs Saúde (ES)'
 )
plt.show()

print("\n🔍 Leitura-chave: valores negativos entre saneamento e morbidade indicam")
print("   que mais cobertura de saneamento está associada a menos internações.")

In [ ]:
# --- TESTE DE HIPÓTESE: KRUSKAL-WALLIS ---
# H0 (hipótese nula): a morbidade é igual independente do nível de saneamento
# H1 (hipótese alternativa): a morbidade difere entre grupos de saneamento

stat, p_valor, df_hip = teste_kruskal_wallis_por_tercis(
    df_final,
    'indice_atendimento_total_agua',
    'Taxa_Morbidade_100k_Hab'
 )

print("📊 TESTE DE KRUSKAL-WALLIS")
print(f"   Estatística H: {stat:.4f}")
print(f"   p-valor:       {p_valor:.6f}")
print()
if p_valor < 0.05:
    print("✅ RESULTADO: Diferença estatisticamente significativa (p < 0.05).")
    print("   → Rejeita-se H0: a morbidade NÃO é igual entre grupos de saneamento.")
else:
    print("⚠️ RESULTADO: Sem evidência estatística suficiente (p ≥ 0.05).")
    print("   → Não é possível rejeitar H0 com os dados disponíveis.")

# Boxplot para visualizar
tem_grupos = (
    not df_hip.empty
    and 'grupo_saneamento' in df_hip.columns
    and df_hip['grupo_saneamento'].notna().any()
    and df_hip['grupo_saneamento'].nunique() >= 2
    and not np.isnan(p_valor)
 )
if tem_grupos:
    plotar_boxplot_kruskal(df_hip, 'Taxa_Morbidade_100k_Hab', p_valor=p_valor)
    plt.show()
else:
    print("⚠️ Boxplot ignorado: grupos insuficientes para plotagem.")